# SigLIP2 美学头蒸馏 (Aesthetic Head Distillation)

在 **不改动 backbone** 的前提下，为你的两个 SigLIP2 模型各蒸馏一个轻量美学评分头：

| STUDENT_ID | hidden | 你项目里 |
|---|---|---|
| `google/siglip2-base-patch16-224` | 768 | `siglip2-base-patch16-224` |
| `google/siglip2-so400m-patch14-384` | 1152 | `siglip2-so400m-patch14-384` |

**思路**：teacher (aesthetic-predictor-v2-5, 1–10 分) 对一批图片打分 → 同一批图片过 *frozen* SigLIP2 vision tower 取 `pooler_output`（即你 runtime 导出的 `image_features`）→ 训一个 `L2norm → MLP → 1` 的头去回归 teacher 分数。

头吃的就是你已经导出的 `image_features`，所以推理时 **复用现有 embedding，一次前向同时出 embedding + 美学分**，不需要第二座塔。

**用法**：选 GPU runtime (A100/H100) → 设好下面 CONFIG 的 `STUDENT_ID` → `Run all`。跑完换另一个 backbone 再 `Run all` 一次（teacher 分数缓存会复用，第二次只重算 embedding）。

> teacher 想换成 SOTA 的 **Q-Align (`q-future/one-align`)** 可以，但它依赖旧版 `transformers`，与 SigLIP2 (`>=4.49`) 冲突，需单独环境分两阶段跑——见最后一节说明。默认 v2-5 是为了一次跑通。

## 1. 安装依赖

In [ ]:
%pip install -q -U "transformers>=4.49" accelerate safetensors
%pip install -q "aesthetic-predictor-v2-5" datasets pillow scipy onnx onnxruntime tqdm
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime (A100/H100).'

## 2. CONFIG —— 唯一需要改的地方

In [ ]:
# === 选一个 backbone（跑完换另一个再 Run all）===
STUDENT_ID = "google/siglip2-so400m-patch14-384"   # 或 "google/siglip2-base-patch16-224"

# === 蒸馏语料 ===
# 默认流式拉取 COCO (parquet 格式, 带干净 image 列, ~11.8w 张自然照片, streaming 不会有 cast 报错)。
# 想用你自己的照片分布（对“照片精选”最贴）就把 LOCAL_IMAGE_DIR 指到一个图片文件夹（如挂载的 Google Drive）。
HF_STREAM_DATASET = "detection-datasets/coco"
HF_STREAM_SPLIT   = "train"
HF_IMAGE_COL      = "image"
LOCAL_IMAGE_DIR   = None        # e.g. "/content/drive/MyDrive/photos"；非 None 时优先用本地图片

N_IMAGES   = 40000              # 蒸馏样本数；A100/H100 上 4w 张约 15-30 分钟
BATCH      = 64                 # 预处理/打分 batch
EPOCHS     = 60                 # 训头（头很小，多跑几轮+早停）
HEAD_LR    = 1e-3
VAL_FRAC   = 0.1
SEED       = 42

WORK = "/content/aesthetic_distill"
import os; os.makedirs(WORK, exist_ok=True)
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda"
SHORT = STUDENT_ID.split('/')[-1]
print('student =', STUDENT_ID, '| work dir =', WORK)

## 3. 加载 teacher (aesthetic-predictor-v2-5) 与 student (frozen SigLIP2)

In [ ]:
from aesthetic_predictor_v2_5 import convert_v2_5_from_siglip

teacher, teacher_proc = convert_v2_5_from_siglip(low_cpu_mem_usage=True, trust_remote_code=True)
teacher = teacher.to(torch.bfloat16).to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

@torch.inference_mode()
def teacher_score(pil_batch):
    px = teacher_proc(images=pil_batch, return_tensors="pt").pixel_values.to(DEVICE, torch.bfloat16)
    logits = teacher(px).logits.squeeze(-1)   # ~1..10
    return logits.float().cpu()

In [ ]:
from transformers import AutoModel, AutoProcessor

student = AutoModel.from_pretrained(STUDENT_ID).to(DEVICE).eval()
student_proc = AutoProcessor.from_pretrained(STUDENT_ID)
for p in student.parameters():
    p.requires_grad_(False)
HIDDEN = student.config.vision_config.hidden_size
print('student pooled (image_features) dim =', HIDDEN)

@torch.inference_mode()
def student_embed(pil_batch):
    inp = student_proc(images=pil_batch, return_tensors="pt")
    vis = {k: v.to(DEVICE) for k, v in inp.items() if k.startswith('pixel') or k in ('spatial_shapes',)}
    pooled = student.vision_model(**vis).pooler_output   # == 你 runtime 的 image_features
    return pooled.float().cpu()

## 4. 语料迭代器（本地文件夹优先，否则流式公共数据集；坏图自动跳过）

In [ ]:
from PIL import Image
import glob, itertools, io
Image.MAX_IMAGE_PIXELS = None

def _to_pil(img):
    # HF 图片列可能是 PIL.Image，也可能是 {'bytes':..,'path':..} 这种 dict
    if isinstance(img, dict):
        if img.get('bytes'):
            img = Image.open(io.BytesIO(img['bytes']))
        elif img.get('path'):
            img = Image.open(img['path'])
        else:
            return None
    return img.convert('RGB')

def image_iter():
    if LOCAL_IMAGE_DIR:
        paths = []
        for ext in ('jpg','jpeg','png','webp','bmp','JPG','JPEG','PNG','WEBP'):
            paths += glob.glob(os.path.join(LOCAL_IMAGE_DIR, '**', f'*.{ext}'), recursive=True)
        random.shuffle(paths)
        for p in paths:
            try:
                yield Image.open(p).convert('RGB')
            except Exception:
                continue
    else:
        from datasets import load_dataset
        ds = load_dataset(HF_STREAM_DATASET, split=HF_STREAM_SPLIT, streaming=True)
        for ex in ds:
            img = ex.get(HF_IMAGE_COL) or ex.get('image') or ex.get('jpg')
            if img is None:
                continue
            try:
                out = _to_pil(img)
                if out is not None:
                    yield out
            except Exception:
                continue

def batched(it, n):
    while True:
        chunk = list(itertools.islice(it, n))
        if not chunk:
            return
        yield chunk

## 5. 预计算 (teacher 分数 + student embedding)，带磁盘缓存

缓存按 backbone 命名；teacher 分数与 embedding 一起存。换 backbone 重跑时若已存在缓存会直接跳过。

In [ ]:
from tqdm.auto import tqdm
cache_path = os.path.join(WORK, f'pairs_{SHORT}.pt')

if os.path.exists(cache_path):
    blob = torch.load(cache_path)
    EMB, SCORE = blob['emb'], blob['score']
    print('loaded cache:', cache_path, EMB.shape, SCORE.shape)
else:
    embs, scores, done = [], [], 0
    pbar = tqdm(total=N_IMAGES)
    for chunk in batched(image_iter(), BATCH):
        try:
            s = teacher_score(chunk)
            e = student_embed(chunk)
        except Exception as ex:
            print('skip batch:', ex); continue
        scores.append(s); embs.append(e)
        done += len(chunk); pbar.update(len(chunk))
        if done >= N_IMAGES:
            break
    pbar.close()
    EMB = torch.cat(embs)[:N_IMAGES]
    SCORE = torch.cat(scores)[:N_IMAGES]
    torch.save({'emb': EMB, 'score': SCORE, 'student': STUDENT_ID}, cache_path)
    print('cached', EMB.shape, '->', cache_path)

print('teacher score range %.2f .. %.2f (mean %.2f)' % (SCORE.min(), SCORE.max(), SCORE.mean()))

## 6. 定义头 + 训练 (frozen backbone, 只训这个 MLP)

头结构沿用 v2-5 风格：`L2norm → 1024 → 128 → 64 → 16 → 1`。L2 归一化**内置在头里**，所以推理时直接喂原始 `image_features` 即可。

In [ ]:
import torch.nn as nn
from scipy.stats import spearmanr, pearsonr

class AestheticHead(nn.Module):
    """输入 = 原始 SigLIP2 image_features (pooler_output)。内部 L2 归一化后过 MLP 输出标量分数。"""
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(1024, 128),    nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),      nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 16),       nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, image_features):
        x = image_features / image_features.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        return self.net(x).squeeze(-1)

# train/val split
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(EMB), generator=g)
n_val = int(len(EMB) * VAL_FRAC)
val_idx, tr_idx = perm[:n_val], perm[n_val:]
Xtr, Ytr = EMB[tr_idx].to(DEVICE), SCORE[tr_idx].to(DEVICE)
Xva, Yva = EMB[val_idx].to(DEVICE), SCORE[val_idx].to(DEVICE)

head = AestheticHead(HIDDEN).to(DEVICE)
opt = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
lossf = nn.MSELoss()
bs = 4096
best_srcc, best_state = -1.0, None

for ep in range(EPOCHS):
    head.train()
    idx = torch.randperm(len(Xtr), device=DEVICE)
    for i in range(0, len(idx), bs):
        b = idx[i:i+bs]
        opt.zero_grad()
        loss = lossf(head(Xtr[b]), Ytr[b])
        loss.backward(); opt.step()
    sched.step()
    head.eval()
    with torch.inference_mode():
        pred = head(Xva).cpu().numpy()
    gt = Yva.cpu().numpy()
    srcc = spearmanr(pred, gt).statistic
    plcc = pearsonr(pred, gt)[0]
    mse = float(((pred - gt) ** 2).mean())
    if srcc > best_srcc:
        best_srcc = srcc
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f'ep {ep:02d}  val SRCC {srcc:.4f}  PLCC {plcc:.4f}  MSE {mse:.4f}')

head.load_state_dict(best_state)
print(f'\nBEST val SRCC = {best_srcc:.4f}  (越接近 1 说明头复刻 teacher 排序越好)')

## 7. 导出：safetensors + config.json + ONNX (头吃原始 image_features)

In [ ]:
import json
from safetensors.torch import save_file

out_dir = os.path.join(WORK, f'aesthetic-head-{SHORT}')
os.makedirs(out_dir, exist_ok=True)
head.eval().cpu()

save_file(head.state_dict(), os.path.join(out_dir, 'head.safetensors'))

config = {
    'backbone': STUDENT_ID,
    'input': 'image_features (SigLIP2 vision pooler_output)',
    'in_dim': HIDDEN,
    'l2_normalize_input': True,
    'mlp': [HIDDEN, 1024, 128, 64, 16, 1],
    'teacher': 'aesthetic-predictor-v2-5',
    'score_range': '~1..10 (teacher scale)',
    'val_srcc': round(float(best_srcc), 4),
    'n_images': int(len(EMB)),
}
with open(os.path.join(out_dir, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

# ONNX: 输入 image_features [B, HIDDEN] -> aesthetic_score [B]
dummy = torch.randn(1, HIDDEN)
torch.onnx.export(
    head, dummy, os.path.join(out_dir, 'aesthetic.onnx'),
    input_names=['image_features'], output_names=['aesthetic_score'],
    dynamic_axes={'image_features': {0: 'batch'}, 'aesthetic_score': {0: 'batch'}},
    opset_version=17,
)
print('exported ->', out_dir)
print(json.dumps(config, indent=2))

## 8. 端到端校验 + ONNX 数值一致性

In [ ]:
import onnxruntime as ort

# 取几张验证图，对比 teacher vs student-head 分数
sample = list(itertools.islice(image_iter(), 8))
with torch.inference_mode():
    t = teacher_score(sample)
    emb = student_embed(sample)
    pt = head(emb).numpy()

sess = ort.InferenceSession(os.path.join(out_dir, 'aesthetic.onnx'))
ox = sess.run(None, {'image_features': emb.numpy()})[0]

print('teacher  :', [f'{v:.2f}' for v in t.tolist()])
print('head(pt) :', [f'{v:.2f}' for v in pt.tolist()])
print('head(onnx):', [f'{v:.2f}' for v in ox.tolist()])
print('onnx vs pt max abs diff =', float(abs(ox - pt).max()))

In [ ]:
# 打包下载（或保存到挂载的 Drive）
import shutil
zip_path = shutil.make_archive(out_dir, 'zip', out_dir)
print('zip ->', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass

## 接入 lumen-hub（导出后）

`aesthetic.onnx` 的输入就是你 vision 已导出的 `image_features`，所以新增一个 task 复用现有 embedding 输出即可，无需再跑 vision tower：

```jsonc
// model_info.json -> task_metadata.tasks
"aesthetic_score": {
  "component": "vision",
  "input_names": ["image_features"],   // 复用 semantic_image_embed 的输出
  "output_name": "aesthetic_score"
}
```

L2 归一化已经 bake 进 ONNX，runtime 直接把原始 `image_features` 喂进去即可。分数范围 ~1–10。

**两个 backbone**：把 `STUDENT_ID` 换成 `google/siglip2-base-patch16-224` 再 `Run all`，得到第二个 `aesthetic.onnx`（768 维输入）。

---
### 可选：换 Q-Align 当 teacher（更强，但要分两阶段）
Q-Align (`q-future/one-align`) 依赖旧版 `transformers`，与 SigLIP2 冲突，**不能同一环境**。做法：
1. 新建一个 Colab，`pip install transformers==4.36.1`，用 `model.score(imgs, task_="aesthetics")` 给同一批图片打分（1–5），存成 `{image_id: score}`。
2. 回到本 notebook，把第 5 节的 `teacher_score` 换成查表读 Q-Align 分数，其余不变。

排序指标 (SRCC) 是 scale-invariant，所以 1–5 还是 1–10 不影响训练，只是输出量纲不同。